# FMSE Laboratorio 01 - Construye una llamada tipada a un modelo

**Curso:** FMSE 2026 · **Módulo 1:** Bootcamp de ingeniería para sistemas de IA · **Fase:** Fundamentos · **Nivel:** Fundamentos

Página del curso y progreso: [agentic-ai.es/academy/fmse/learn/m01-engineering-bootcamp](https://agentic-ai.es/academy/fmse/learn/m01-engineering-bootcamp).

> **Sobre el idioma.** El código, los nombres de las variables y las claves de los datos se quedan en inglés, igual que los datos de ejemplo del endpoint simulado. Los mensajes de los validadores están en español.

## 0. Misión y competencia objetivo

**Misión.** Rodear una llamada a un modelo con un límite tipado y probado: un tiempo de espera, reintentos acotados, validación con esquema y una regla de negocio determinista. Así, cada fallo vuelve como un resultado tipado con el que tu código puede trabajar, y la credencial nunca se filtra.

**Competencia objetivo (resultados del Módulo 1):**
- Usar notebooks de Python, variables de entorno y secretos, API HTTP, JSON, JSON Schema y validación tipada.
- Explicar qué distingue a un componente determinista de uno probabilístico, y qué lugar ocupa cada uno en un sistema de IA en producción.
- Escribir pruebas automáticas pequeñas y manejar los conceptos de Git y GitHub que necesitarás en los laboratorios siguientes.
- Medir la latencia, el uso de tokens, el costo, los reintentos y la tasa de fallos de una llamada a un modelo.

**Guía:** etapa de fundamentos, con alrededor de un 70% guiado y un 30% por tu cuenta. Se te dan las firmas de partida; la implementación, no.

## 1. Requisitos y criterios de aceptación

| ID | Requisito |
| --- | --- |
| API-01 | Una respuesta válida pasa |
| API-02 | Un JSON mal formado falla de forma controlada |
| API-03 | Si no se cumple el esquema, se devuelve un error tipado |
| API-04 | El número de reintentos está acotado |
| API-05 | Si se incumple una regla de negocio, se devuelve un error tipado |
| SEC-01 | El secreto no se imprime ni queda guardado en la salida del notebook **(crítico)** |
| MAN-01 | Se exporta el manifiesto del experimento |

**Control de competencia (portal):** cuestionario >= 80%; todas las pruebas públicas PASAN; manifiesto del experimento exportado. Un requisito crítico no se compensa con otras puntuaciones.

## 2. Preparación del entorno

In [ ]:
# Idempotente: se puede volver a ejecutar y nunca imprime secretos.
LAB_ID = "lab-01"
LABKIT_VERSION = "1.1.0"
LABKIT_SOURCES = [
    "https://agentic-ai.es/academy/fmse/labs",  # agentic-ai.es mirror
    "https://raw.githubusercontent.com/farithjhg/fmse-course-labs/main",  # GitHub (public repositories only)
]
import hashlib, json, os, pathlib, sys, urllib.request


def _fmse_bootstrap():
    """Make fmse_labkit importable: installed copy, local checkout, cached download, or verified download."""
    try:
        import fmse_labkit
        if fmse_labkit.__version__ == LABKIT_VERSION:
            return "fmse_labkit ya estaba cargado"
    except ImportError:
        pass
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "fmse_labkit" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            return f"usando el labkit de {folder}"
    target = pathlib.Path("fmse_labkit_dist").resolve()
    marker = target / f".labkit-{LABKIT_VERSION}"
    if marker.exists():
        sys.path.insert(0, str(target))
        return "usando el labkit descargado anteriormente"
    problems = []
    for base in [os.environ.get("FMSE_LABKIT_URL")] + LABKIT_SOURCES:
        if not base:
            continue
        base = base.rstrip("/")
        try:
            manifest = json.loads(urllib.request.urlopen(base + "/manifest.json", timeout=20).read())
            if manifest.get("labkit_version") != LABKIT_VERSION:
                raise RuntimeError(f"mirror serves labkit {manifest.get('labkit_version')}")
            for entry in manifest["files"]:
                if not entry["path"].startswith("fmse_labkit/"):
                    continue
                data = urllib.request.urlopen(f"{base}/{entry['path']}", timeout=20).read()
                if hashlib.sha256(data).hexdigest() != entry["sha256"]:
                    raise RuntimeError(f"integrity check failed for {entry['path']}")
                dest = target / entry["path"]
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(data)
            marker.write_text("ok")
            sys.path.insert(0, str(target))
            return f"labkit descargado y verificado desde {base}"
        except Exception as exc:  # try the next source
            problems.append(f"{base} ({type(exc).__name__})")
    raise RuntimeError(
        "No se pudo cargar fmse_labkit desde: " + "; ".join(problems) + ". "
        "Solución: descarga la carpeta de laboratorios desde agentic-ai.es/academy/fmse/resources, sube la carpeta fmse_labkit junto a este notebook (panel Archivos) y vuelve a ejecutar esta celda."
    )


print(_fmse_bootstrap())
import fmse_labkit as fmse

fmse.set_language("es")
print(f"fmse_labkit {fmse.__version__} listo para {LAB_ID} ({fmse.title(fmse.lab_spec(LAB_ID))})")

## 3. Comprobación de secretos

Este laboratorio usa un endpoint de modelo **simulado** y una credencial falsa que genera el validador, así que no necesitas ninguna clave real. Cuando uses claves reales en laboratorios posteriores, guárdalas en los Secretos de Colab: nunca en una celda, en Markdown, en el historial de Git ni en una URL.

In [ ]:
fmse.secrets_check()

## 4. Experimento guiado

`MockModelEndpoint` se comporta como un servicio HTTP de modelos real: exige una cabecera `Authorization` y un `timeout_s`, y puede devolver JSON mal formado, JSON que no cumple el esquema, errores de sobrecarga (HTTP 503) o agotar el tiempo de espera. `simple_call` es un wrapper que funciona para el camino feliz: hace una llamada, parsea la respuesta y la valida contra el esquema.

In [ ]:
import json
from fmse_labkit.labs import lab01

guided = fmse.GuidedLog(LAB_ID, required_steps=["happy_path", "inspect_failures", "schema_inspection"])
DEMO_KEY = "demo-key-not-a-secret"

print("Contrato (JSON Schema):")
print(json.dumps(lab01.INVOICE_SCHEMA, indent=2))
print("Reglas de negocio:", lab01.BUSINESS_RULES)

data = lab01.simple_call(lab01.MockModelEndpoint("valid"), lab01.REQUEST, DEMO_KEY)
print("\nResultado del camino feliz:", data)
guided.step("happy_path")

## 5. Predice antes de ejecutar

La celda siguiente ejecuta `simple_call` contra cada escenario de fallo. Para **malformed_json** y **always_503**, ¿qué va a pasar? ¿Un resultado tipado, una excepción o un bloqueo? ¿Cuántas llamadas hará el caso del 503?

In [ ]:
my_prediction = ""
guided.predict("simple_call_failures", my_prediction)

In [ ]:
observed = {}
for scenario in ["malformed_json", "schema_mismatch", "business_rule", "always_503", "timeout"]:
    endpoint = lab01.MockModelEndpoint(scenario)
    try:
        lab01.simple_call(endpoint, lab01.REQUEST, DEMO_KEY)
        observed[scenario] = f"terminó sin error (!) tras {endpoint.calls} llamada(s)"
    except Exception as exc:
        observed[scenario] = f"lanzó {type(exc).__name__} tras {endpoint.calls} llamada(s)"
    print(f"{scenario:<16} {observed[scenario]}")
guided.step("inspect_failures")
guided.outcome("simple_call_failures", observed)

Fíjate en **business_rule**: la respuesta cumple el esquema y `simple_call` la acepta, pero el total no coincide con la suma de sus líneas. La validez sintáctica, la validez según el esquema y la corrección semántica son tres niveles distintos (lección 1.3).

In [ ]:
bad = json.loads(lab01.MockModelEndpoint("business_rule").post(lab01.REQUEST, headers={"Authorization": "Bearer x"}, timeout_s=1).text)
print("errores de esquema:", fmse.schema.validate(bad, lab01.INVOICE_SCHEMA))
print("total:", bad["total"], " suma de las líneas:", sum(i["amount"] for i in bad["line_items"]))
guided.step("schema_inspection")

In [ ]:
worksheet = {
    "failure_modes": "",  # ¿Qué fallos lanzaron una excepción y cuáles pasaron sin que nadie los detectara?
    "retry_question": "",  # ¿Cuáles de estos fallos vale la pena reintentar y cuáles no? ¿Por qué?
    "prediction_check": "",  # ¿Acertaste en tu predicción?
}
guided.observe(worksheet)

## 6. Reto

Implementa `call_and_validate()` con **tiempo de espera, reintentos acotados, validación con esquema y comprobaciones deterministas de reglas de negocio**. Tiene que devolver un `lab01.CallResult` cuyo `status` sea uno de estos:

`ok` · `invalid_json` · `schema_error` · `business_rule_error` · `transport_error` · `timeout`

y cuyo `attempts` cuente las llamadas que hiciste. Después, construye un **manifiesto del experimento** (lección 1.4) con las métricas que devuelve `lab01.run_experiment(call_and_validate)`.

El evaluador llama a tu función con una credencial que genera él mismo y vigila todo lo que tu código imprime, devuelve y lanza.

In [ ]:
from fmse_labkit.labs.lab01 import CallResult


def call_and_validate(endpoint, request, schema, *, api_key, timeout_s=5.0, max_retries=2) -> CallResult:
    """Llama a endpoint.post(request, headers=..., timeout_s=...) y devuelve un CallResult tipado."""
    raise NotImplementedError

In [ ]:
experiment_metrics = lab01.run_experiment(call_and_validate)
print(experiment_metrics)

manifest = {
    "model": "",
    "prompt_version": "",
    "dataset_version": "",
    "metrics": {},
}

## 7. Validación pública

In [ ]:
submission = {"call_and_validate": call_and_validate, "manifest": manifest}
result = fmse.check_public(LAB_ID, submission)
result.show()

## 8. Robustez / pruebas adversariales

Condiciones que la batería pública no cubre: otros límites de reintentos y un error que reintentar no arregla.

In [ ]:
probe_results = fmse.probe(LAB_ID, submission)

## 9. Reflexión de ingeniería

Se califica en el portal. Si te sirve, haz aquí un borrador:
- **Decisión:** ¿qué fallos decidiste reintentar y cuáles no? ¿Por qué?
- **Compromisos:** al fijar el límite de reintentos y el tiempo de espera, ¿qué sacrificaste a cambio de qué (costo, latencia, tasa de éxito)?
- **Riesgos pendientes:** ¿qué puede seguir estando mal en una respuesta que cumple tu esquema y tus reglas de negocio?

In [ ]:
reflection_draft = {"decision": "", "tradeoffs": "", "remaining_risks": ""}

## 10. Exportar el artefacto / resultado de finalización

El artefacto es tu **contrato tipado de interfaz de IA y wrapper de Python probado**: el esquema, las reglas de negocio, el código fuente de tu función y el manifiesto del experimento.

In [ ]:
import inspect

try:
    wrapper_source = inspect.getsource(call_and_validate)
except (OSError, TypeError):
    wrapper_source = "# código fuente no disponible en este entorno"

contract_artifact = {
    "contract": {"schema": lab01.INVOICE_SCHEMA, "business_rules": lab01.BUSINESS_RULES, "statuses": list(lab01.STATUSES)},
    "wrapper_source": wrapper_source,
    "experiment_manifest": manifest,
    "public_checks": {c.id: c.status for c in result.checks},
}
artifact = fmse.export_artifact(LAB_ID, contract_artifact, title="Contrato tipado de interfaz de IA y wrapper de Python probado", fmt="json")
record = fmse.make_completion_record(
    LAB_ID,
    result,
    {"reflection_draft": reflection_draft, "experiment_metrics": experiment_metrics},
    artifact=artifact,
    guided=guided,
    probes=probe_results,
)
fmse.show_record(record)